# Transformer Sentence Embeddings - Kaggle Training

## Pipeline (run cells top-to-bottom)

| Phase | Cell | What it does | Output |
|-------|------|--------------|--------|
| 1 | 4 | MNR training on AllNLI pairs | `best_model.pt`, `vocab.pkl` |
| 2 | 4 | Evaluate best model on sts-222 val + test | `eval_results.json` |
| 3 | 4 | Triplet fine-tuning on AllNLI | `final_triplet_model.pt` |
| Display | 5 | Print evaluation metrics | — |
| Demo | 6 | Inference similarity demo | — |

## Before running
1. **Upload project** as a Kaggle dataset (containing `train.py`, `data.py`, `models/`, `losses/`, `evaluate_search.py`) and set `PROJECT_DIR` in Cell 2.
2. **Attach sts-222 dataset** (e.g. `amreisa9/sts-222`) and set `STS222_DIR` in Cell 3.
3. **AllNLI** is downloaded automatically from HuggingFace — no manual upload needed.
4. **Enable GPU**: Settings → Accelerator → GPU T4
5. **Enable Internet**: Settings → Internet → On (required for AllNLI download)

In [ ]:
!pip install -q datasets scipy tqdm

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import os, sys, shutil

# ── Set to your Kaggle dataset slug containing the project files ──────────────
PROJECT_DIR = '/kaggle/input/transformer-sbert'   # <-- update this
# ─────────────────────────────────────────────────────────────────────────────

WORKING_DIR = '/kaggle/working'

if os.path.isdir(PROJECT_DIR):
    print(f'Copying project files from {PROJECT_DIR} ...')
    for item in os.listdir(PROJECT_DIR):
        src = os.path.join(PROJECT_DIR, item)
        dst = os.path.join(WORKING_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print('Done.')
else:
    print(f'[WARNING] PROJECT_DIR not found: {PROJECT_DIR}')
    print('Files must already be present in the working directory.')

sys.path.insert(0, WORKING_DIR)
print(f'sys.path[0] = {WORKING_DIR}')
print('Files:', [f for f in os.listdir(WORKING_DIR) if not f.startswith('.')][:20])

In [ ]:
import os, shutil, pandas as pd

os.makedirs('data/AllNLI', exist_ok=True)
os.makedirs('data/sts-222', exist_ok=True)

# ── AllNLI (training data) ────────────────────────────────────────────────────
# Upload your local AllNLI.csv (277,230 rows) as a Kaggle dataset and set
# ALLNLI_KAGGLE to the mounted path.  The HuggingFace download only returns a
# small curated subset — use the full file for best results.
ALLNLI_KAGGLE = '/kaggle/input/allnli-dataset/AllNLI.csv'   # <-- update this
ALLNLI_PATH   = 'data/AllNLI/AllNLI.csv'

if os.path.exists(ALLNLI_PATH):
    n_rows = len(pd.read_csv(ALLNLI_PATH))
    print(f'AllNLI already present ({n_rows:,} rows).')
elif os.path.exists(ALLNLI_KAGGLE):
    shutil.copy(ALLNLI_KAGGLE, ALLNLI_PATH)
    n_rows = len(pd.read_csv(ALLNLI_PATH))
    print(f'Copied AllNLI from Kaggle dataset ({n_rows:,} rows).')
else:
    # Fallback: download from HuggingFace (gives a smaller subset ~48k)
    print('[WARNING] ALLNLI_KAGGLE not found. Downloading from HuggingFace ...')
    print('          This gives only ~48k pairs. Upload the full 277k file for best results.')
    from datasets import load_dataset
    ds = load_dataset('sentence-transformers/all-nli', 'triplet', split='train')
    df = pd.DataFrame({'anchor': ds['anchor'], 'positive': ds['positive'], 'negative': ds['negative']})
    df.to_csv(ALLNLI_PATH, index=False)
    print(f'Downloaded {len(df):,} AllNLI triplets -> {ALLNLI_PATH}.')

# ── sts-222 (evaluation data) ─────────────────────────────────────────────────
STS222_DIR = '/kaggle/input/datasets/amreisa9/sts-222'   # <-- update this

for fname in ('stsb_validation.csv', 'stsb_test.csv'):
    dst = f'data/sts-222/{fname}'
    if os.path.exists(dst):
        print(f'{dst} already present.')
        continue
    src = os.path.join(STS222_DIR, fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {fname}.')
    else:
        print(f'[WARNING] {src} not found — check STS222_DIR.')

print('
Data ready.')


In [ ]:
import train

# Optional: override CONFIG values before training starts.
# train.CONFIG['mnr_epochs']    = 10
# train.CONFIG['batch_size']    = 32
# train.CONFIG['triplet_epochs'] = 3

# Runs all 3 phases in sequence:
#   Phase 1 — MNR training on AllNLI pairs         -> best_model.pt, vocab.pkl
#   Phase 2 — Evaluate best model on sts-222        -> eval_results.json
#   Phase 3 — Triplet fine-tuning on AllNLI         -> final_triplet_model.pt
train.main()


In [ ]:
# ── Show evaluation results (Phase 2 output) ──────────────────────────────────
import json

with open('eval_results.json') as f:
    results = json.load(f)

print('Evaluation results on sts-222 (best MNR model, before Triplet fine-tuning)\n')
for split, m in results.items():
    print(f'{split.upper()}')
    print(f"  Recall@1  : {m['recall@1']:.4f}")
    print(f"  Recall@5  : {m['recall@5']:.4f}")
    print(f"  Recall@10 : {m['recall@10']:.4f}")
    print(f"  MRR       : {m['mrr']:.4f}")
    print(f"  Spearman  : {m['spearman']:.4f}")
    print()

In [ ]:
# ── Inference demo ────────────────────────────────────────────────────────────
import pickle, torch
from models.model.transformer import Transformer

MODEL_CONFIG = {
    'max_len':  128,
    'd_model':  256,
    'n_layers': 4,
    'n_heads':  4,
    'd_ff':     512,
    'pooling':  'mean',
}

with open('vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)
print(f'Vocab: {len(vocab):,} tokens')

model = Transformer(
    vocab_size=len(vocab),
    d_model=MODEL_CONFIG['d_model'],
    n_layers=MODEL_CONFIG['n_layers'],
    n_heads=MODEL_CONFIG['n_heads'],
    d_ff=MODEL_CONFIG['d_ff'],
    max_len=MODEL_CONFIG['max_len'],
    dropout=0.0,
    pooling=MODEL_CONFIG['pooling'],
).to(DEVICE)
model.load_state_dict(torch.load('final_triplet_model.pt', map_location=DEVICE))
model.eval()
print('Loaded final_triplet_model.pt')

@torch.no_grad()
def embed(text):
    cls_id = vocab.word2idx[vocab.CLS_TOKEN]
    sep_id = vocab.word2idx[vocab.SEP_TOKEN]
    ids = torch.tensor(
        [[cls_id] + vocab.encode(text)[:MODEL_CONFIG['max_len'] - 2] + [sep_id]],
        dtype=torch.long
    ).to(DEVICE)
    return model.encode(ids, normalize=True)

def similarity(t1, t2):
    return (embed(t1) * embed(t2)).sum().item()

test_pairs = [
    ('A dog is running in the park',  'A dog is playing outside'),
    ('The man is eating a sandwich',  'A person is having lunch'),
    ('A woman is singing a song',     'A lady is performing music'),
    ('The cat sat on the mat',        'A dog is running in the park'),
    ('A man is riding a bicycle',     'Someone is driving a car'),
    ('I love pizza',                  'The stock market crashed today'),
]

print(f'\n{"Sentence 1":<42} {"Sentence 2":<42} {"Sim":>5}')
print('-' * 95)
for s1, s2 in test_pairs:
    sim = similarity(s1, s2)
    bar = '#' * int((sim + 1) * 10)
    print(f'{s1:<42} {s2:<42} {sim:>5.3f}  {bar}')